# Notebook: 05 Prediction
### Purpose: load a trained model, run inference, preview overlays, and optionally export a submission JSON.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
# ponytail: PROJECT_ROOT or walk up from notebooks[/explore]
_cwd = Path.cwd()
root = Path(os.getenv("PROJECT_ROOT") or (
    _cwd.parent if _cwd.name == "notebooks"
    else _cwd.parent.parent if _cwd.name == "explore"
    else _cwd
)).resolve()
sys.path[:0] = [str(root), str(root / "src")]

import torch

from src.data.augmentations import get_val_augmentations
from src.exploration.visualize import show_side_by_side
from src.models.zoo import MODEL_BUILDERS
from src.prediction.pipeline import Predictor
from src.prediction.submission import export_submission
from src.utils.config import Config
from src.utils.helpers import c, init_notebook, p
from src.utils.versioning import VersionManager
from src.exploration.evaluation import load_best_model

config = Config.load(root = root)

init_notebook(config.train.seed)



#### Select model version

In [ ]:
p("Models", MODEL_BUILDERS)
#config.show()
p("Batch", config.train.batch_size)
p("Epochs", config.train.epochs)
p("Learning Rate", config.train.learning_rate, precision = 9)
p("Image Size", config.train.image_size)


In [ ]:
model_name = "simple_cnn"

try:
    model = load_best_model(model_name, config, notebook = "03", mode = "rgb")

    # Save model temporarily for Predictor class
    temp_model_path = config.paths.models / "temp_best_model.pth"
    torch.save({ "model": model.state_dict() }, temp_model_path)

    predictor = Predictor(
            model_path = temp_model_path,
            model_name = model_name,
            image_size = config.train.image_size,
    )

    p(f"Successfully created predictor with {model_name}", color1 = c.GREEN)

except RuntimeError as e:
    p("No trained models found!", color1 = c.RED, bold = True)
    p("Available directories:", color1 = c.ORANGE)
    models_dir = config.paths.models
    if models_dir.exists():
        p("Top-level directories:")
        for item in models_dir.iterdir():
            if item.is_dir():
                p("  ", item.name)
    else:
        p("Models directory doesn't exist:", models_dir)

    raise e


#### Run on evaluation folder

In [ ]:
eval_dir = config.paths.eval_images
p("eval_dir", eval_dir)

transform = get_val_augmentations(config.train.image_size)

results = predictor.run_on_folder(eval_dir, transform = transform, num_samples = 10)


#### Visualizations Samples

In [ ]:
for r in results:
    img = r["image"]
    mask = r["mask"]
    overlay = r["overlay"]

    # show_image(img, r["name"])
    # show_mask(mask, "Predicted mask")
    # show_overlay(img, mask, 0.4, "Overlay")

    show_side_by_side(
            img, mask, overlay,
            titles = [r["name"], "Predicted mask", "Overlay"],
            cmaps = [None, "gray", None],
    )


### Export submission file

In [ ]:
vm = VersionManager(config.paths.models)
version_folder = vm.find_latest()

if version_folder is None:
    p("No model version found for export", color1 = c.RED)
else:
    p("Exporting submission for version:", version_folder.name)

    out_path = version_folder / "submission.json"
    export_submission(results, out_path, config)
    p("Submission Saved", out_path)
